
# Independent Daily-Feature State Model

This notebook is a **new independent model**. It does not try to reproduce Daniel's pipeline.

The idea is simple:

1. Convert the available ~15-minute SCADA into a **dense daily feature table**.
2. Learn, from the three labelled turbines, which **within-turbine feature changes** correspond to changes in yaw misalignment.
3. Keep the old physics model only for the **absolute turbine-level yaw centre**.
4. Denoise the learned daily yaw variation into piecewise-constant states.
5. Evaluate everything with strict **leave-one-turbine-out** validation before generating any candidate submission.

The model never uses target labels, teammate predictions, teammate clusters, or teammate change-point dates.

Model structure:

`old physics absolute level + learned SCADA temporal variation`


In [ ]:

from pathlib import Path
import sys
import math
import warnings

import numpy as np
import pandas as pd
from IPython.display import display

from sklearn.linear_model import Ridge

ROOT = Path(r"E:/EnergyHacks/github_release")
SRC = ROOT / "src"
ZIP_PATH = Path(r"E:/EnergyHacks/turbines_data.zip")

if not ROOT.exists():
    raise FileNotFoundError(ROOT)
if not SRC.exists():
    raise FileNotFoundError(SRC)
if not ZIP_PATH.exists():
    raise FileNotFoundError(ZIP_PATH)

sys.path.insert(0, str(SRC))
sys.path.insert(0, str(ROOT))

from baseline_loto_ridge import read_turbine

TRAIN = ["PPP_WTG12", "PPP_WTG13", "PPP_WTG14"]
VALIDATE_TARGET = "PPP_WTG17"
FINAL_TARGET = "SSS_WTG06"

# Frozen long-term physics features from the original standalone model.
THETA_STAR = {
    "PPP_WTG12": -3.441,
    "PPP_WTG13": +0.357,
    "PPP_WTG14": -4.351,
}

# Frozen target absolute levels from the original standalone model.
FROZEN_TARGET_LEVEL = {
    "PPP_WTG17": -3.596,
    "SSS_WTG06": -4.568,
}

ALPHA_GRID = [0.1, 1.0, 10.0, 100.0]
SEG_PENALTY_GRID = [2.0, 5.0, 10.0, 20.0, 40.0]
MIN_STATE_DAYS = 14

print("ROOT:", ROOT)
print("ZIP_PATH:", ZIP_PATH)



## 1. Dense daily SCADA features

The previous short-window power-peak estimator was too sparse for state detection.

This version instead builds features that exist on almost every day:

- daily median and spread of `WindDir - NacDir`;
- daily nacelle movement / yaw activity;
- daily wind-direction variability;
- normalized-power level after removing the wind-speed dependence;
- daily WindSpeed, PitchAngle, RotSpeed and GenSpeed summaries;
- centred rolling summaries and short-vs-long-window contrasts.

Every feature is **centred and scaled within its own turbine**, so the temporal model learns changes relative to that turbine's normal behaviour rather than absolute sensor offsets.


In [ ]:

def wrap_deg(x):
    x = np.asarray(x, dtype=float)
    return (x + 180.0) % 360.0 - 180.0


def find_timestamp_column(df):
    preferred = [
        "timestamp", "Timestamp", "datetime", "DateTime",
        "date_time", "time", "Time", "Date", "date"
    ]

    for c in preferred:
        if c in df.columns:
            parsed = pd.to_datetime(df[c], errors="coerce")
            if parsed.notna().mean() > 0.95:
                return parsed

    if isinstance(df.index, pd.DatetimeIndex):
        return pd.Series(df.index, index=df.index)

    for c in df.columns:
        if "date" in str(c).lower() or "time" in str(c).lower():
            parsed = pd.to_datetime(df[c], errors="coerce")
            if parsed.notna().mean() > 0.95:
                return parsed

    raise ValueError(f"Could not identify timestamp. Columns: {list(df.columns)}")


def circular_resultant_deg(values):
    a = np.asarray(pd.Series(values).dropna(), dtype=float)
    if len(a) == 0:
        return np.nan
    r = np.deg2rad(a)
    return float(np.hypot(np.mean(np.sin(r)), np.mean(np.cos(r))))


def robust_iqr(values):
    a = np.asarray(pd.Series(values).dropna(), dtype=float)
    if len(a) == 0:
        return np.nan
    return float(np.percentile(a, 75) - np.percentile(a, 25))


def build_daily_features(turbine_id):
    raw = read_turbine(str(ZIP_PATH), turbine_id).copy()
    ts = find_timestamp_column(raw)

    needed = [
        "Power", "WindSpeed", "WindDir", "NacDir",
        "PitchAngle", "RotSpeed", "GenSpeed",
    ]
    missing = [c for c in needed if c not in raw.columns]
    if missing:
        raise KeyError(f"{turbine_id}: missing {missing}")

    x = pd.DataFrame({
        "timestamp": pd.to_datetime(ts, errors="coerce").to_numpy(),
    })

    for c in needed:
        x[c] = pd.to_numeric(raw[c], errors="coerce").to_numpy()

    x = (
        x.dropna(subset=["timestamp", "WindDir", "NacDir"])
         .sort_values("timestamp")
         .reset_index(drop=True)
    )

    x["date"] = x["timestamp"].dt.normalize()
    x["gamma"] = wrap_deg(x["WindDir"] - x["NacDir"])

    # Operating-region mask. Kept deliberately simple and relative.
    power98 = x["Power"].quantile(0.98)
    pitch95 = x["PitchAngle"].quantile(0.95)

    op = (
        x["WindSpeed"].between(3, 13)
        & (x["Power"] > 0)
        & (x["RotSpeed"] > 0)
        & (x["GenSpeed"] > 0)
        & (x["Power"] <= power98)
        & (x["PitchAngle"] <= pitch95)
    )

    xo = x.loc[op].copy()

    # Remove the dominant wind-speed effect from power using the entire
    # turbine history, without labels.
    xo["ws_bin"] = np.floor(xo["WindSpeed"]).astype("Int64")
    ws_ref = xo.groupby("ws_bin")["Power"].median()
    xo["power_ref"] = xo["ws_bin"].map(ws_ref)
    xo["power_norm"] = xo["Power"] / xo["power_ref"].replace(0, np.nan)

    # Nacelle movement between consecutive SCADA rows.
    x["nac_step"] = np.abs(
        wrap_deg(
            x["NacDir"].to_numpy()[1:].tolist() + [np.nan]
            if False else x["NacDir"].diff().to_numpy()
        )
    )
    # pandas diff across circular values must be wrapped:
    x["nac_step"] = np.abs(wrap_deg(x["NacDir"].diff().to_numpy()))

    rows = []

    all_dates = pd.date_range(
        x["date"].min(),
        x["date"].max(),
        freq="D",
    )

    for d in all_dates:
        g_all = x[x["date"] == d]
        g = xo[xo["date"] == d]

        if len(g_all) == 0:
            rows.append({"date": d})
            continue

        gamma_vals = g["gamma"].dropna().to_numpy(dtype=float)
        pnorm_vals = g["power_norm"].dropna().to_numpy(dtype=float)

        row = {
            "date": d,
            "n_rows": len(g_all),
            "n_operating": len(g),

            "gamma_med": float(np.median(gamma_vals)) if len(gamma_vals) else np.nan,
            "gamma_iqr": robust_iqr(gamma_vals),

            "wind_resultant": circular_resultant_deg(g_all["WindDir"]),
            "nac_resultant": circular_resultant_deg(g_all["NacDir"]),

            "nac_step_med": float(g_all["nac_step"].median()),
            "nac_step_p90": float(g_all["nac_step"].quantile(0.90)),
            "yaw_move_count": float((g_all["nac_step"] > 0.5).sum()),

            "power_norm_med": float(np.nanmedian(pnorm_vals)) if len(pnorm_vals) else np.nan,
            "power_norm_iqr": robust_iqr(pnorm_vals),

            "ws_med": float(g["WindSpeed"].median()),
            "pitch_med": float(g["PitchAngle"].median()),
            "rot_med": float(g["RotSpeed"].median()),
            "gen_med": float(g["GenSpeed"].median()),
        }

        rows.append(row)

    daily = pd.DataFrame(rows).sort_values("date").reset_index(drop=True)

    base_features = [
        "gamma_med", "gamma_iqr",
        "wind_resultant", "nac_resultant",
        "nac_step_med", "nac_step_p90", "yaw_move_count",
        "power_norm_med", "power_norm_iqr",
        "ws_med", "pitch_med", "rot_med", "gen_med",
    ]

    # Fill small gaps from neighbouring days; the raw features are dense.
    for c in base_features:
        daily[c] = (
            daily[c]
            .interpolate(limit=7, limit_direction="both")
        )

    # Rolling temporal context.
    for c in base_features:
        r7 = daily[c].rolling(7, center=True, min_periods=3).median()
        r21 = daily[c].rolling(21, center=True, min_periods=7).median()

        daily[f"{c}_r7"] = r7
        daily[f"{c}_r21"] = r21
        daily[f"{c}_contrast"] = r7 - r21

    feature_cols = [
        c for c in daily.columns
        if c not in {"date", "n_rows", "n_operating"}
    ]

    # Per-turbine robust centring/scaling: learn temporal deviations, not
    # absolute turbine sensor offsets.
    for c in feature_cols:
        med = daily[c].median()
        q1 = daily[c].quantile(0.25)
        q3 = daily[c].quantile(0.75)
        scale = q3 - q1

        if not np.isfinite(scale) or scale < 1e-8:
            scale = daily[c].std()

        if not np.isfinite(scale) or scale < 1e-8:
            scale = 1.0

        daily[c] = (daily[c] - med) / scale

    # Final finite fill after scaling.
    daily[feature_cols] = (
        daily[feature_cols]
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0.0)
    )

    return daily, feature_cols


feature_cache = {}
FEATURE_COLS = None

for t in TRAIN + [VALIDATE_TARGET, FINAL_TARGET]:
    daily, cols = build_daily_features(t)
    feature_cache[t] = daily

    if FEATURE_COLS is None:
        FEATURE_COLS = cols
    else:
        assert FEATURE_COLS == cols

    print(
        t,
        "days =", len(daily),
        "features =", len(FEATURE_COLS),
    )



## 2. Learn temporal yaw variation

The labelled target for this model is **not the absolute yaw level**.

For each labelled turbine, we subtract its own mean yaw value and learn only the day-to-day / state-to-state deviation around that mean.

This prevents the supervised model from replacing the original physics-based cross-turbine calibration.

After prediction, the temporal component is re-centred to zero and added to the physics-derived absolute level.


In [ ]:

def daily_labels(turbine_id):
    raw = read_turbine(str(ZIP_PATH), turbine_id).copy()
    ts = find_timestamp_column(raw)

    if "yaw_misalignment_deg" not in raw.columns:
        return None

    z = pd.DataFrame({
        "date": pd.to_datetime(ts, errors="coerce").dt.normalize().to_numpy(),
        "target": pd.to_numeric(
            raw["yaw_misalignment_deg"],
            errors="coerce",
        ).to_numpy(),
    }).dropna()

    return (
        z.groupby("date", as_index=False)["target"]
         .median()
         .sort_values("date")
    )


label_cache = {t: daily_labels(t) for t in TRAIN}


def l2_segment(values, penalty, min_len=14):
    y = np.asarray(values, dtype=float)
    n = len(y)

    s1 = np.r_[0.0, np.cumsum(y)]
    s2 = np.r_[0.0, np.cumsum(y * y)]

    def cost(i, j):
        m = j - i
        sy = s1[j] - s1[i]
        sy2 = s2[j] - s2[i]
        return sy2 - sy * sy / m

    dp = np.full(n + 1, np.inf)
    prev = np.full(n + 1, -1, dtype=int)
    dp[0] = -penalty

    for end in range(min_len, n + 1):
        for start in range(0, end - min_len + 1):
            if start != 0 and start < min_len:
                continue
            if not np.isfinite(dp[start]):
                continue

            v = dp[start] + cost(start, end) + penalty

            if v < dp[end]:
                dp[end] = v
                prev[end] = start

    if prev[n] < 0:
        return [(0, n)]

    spans = []
    end = n

    while end > 0:
        start = int(prev[end])
        spans.append((start, end))
        end = start

    return spans[::-1]


def label_state_table(turbine_id):
    lab = label_cache[turbine_id].copy()

    full = pd.DataFrame({
        "date": pd.date_range(
            lab["date"].min(),
            lab["date"].max(),
            freq="D",
        )
    }).merge(lab, on="date", how="left")

    y = full["target"].interpolate(limit_direction="both")
    spans = l2_segment(y.to_numpy(), penalty=20.0, min_len=7)

    state = np.empty(len(full), dtype=int)
    state_level = np.empty(len(full), dtype=float)
    boundaries = []

    for k, (i, j) in enumerate(spans):
        level = float(np.median(y.iloc[i:j]))
        state[i:j] = k
        state_level[i:j] = level
        if k > 0:
            boundaries.append(full["date"].iloc[i])

    full["true_state"] = state
    full["state_level"] = state_level
    full["scored"] = full["target"].notna()

    for b in boundaries:
        full.loc[
            (full["date"] >= b - pd.Timedelta(days=7))
            & (full["date"] <= b + pd.Timedelta(days=7)),
            "scored"
        ] = False

    full.loc[
        (full["target"] - full["state_level"]).abs() > 0.75,
        "scored"
    ] = False

    return full


label_state_cache = {
    t: label_state_table(t)
    for t in TRAIN
}


def target_mean(t):
    return float(label_cache[t]["target"].mean())


def calibration_C(train_ids):
    return float(np.mean([
        target_mean(t) + THETA_STAR[t]
        for t in train_ids
    ]))


def physics_level(turbine_id, train_ids):
    return calibration_C(train_ids) - THETA_STAR[turbine_id]


def training_rows(turbines):
    frames = []

    for t in turbines:
        f = feature_cache[t]
        lab = label_cache[t]

        z = f.merge(lab, on="date", how="inner").dropna(subset=["target"])
        z["target_rel"] = z["target"] - z["target"].mean()
        z["turbine"] = t
        frames.append(z)

    return pd.concat(frames, ignore_index=True)


def fit_ridge(train_ids, alpha):
    z = training_rows(train_ids)

    model = Ridge(
        alpha=float(alpha),
        fit_intercept=True,
    )

    model.fit(
        z[FEATURE_COLS].to_numpy(dtype=float),
        z["target_rel"].to_numpy(dtype=float),
    )

    return model


def raw_temporal_prediction(model, turbine_id):
    f = feature_cache[turbine_id]
    pred = model.predict(
        f[FEATURE_COLS].to_numpy(dtype=float)
    )

    # Robust smoothing and zero-centering: only temporal structure remains.
    s = pd.Series(pred, index=f["date"])
    s = s.rolling(7, center=True, min_periods=3).median()
    s = s.interpolate(limit_direction="both")

    s = s - np.average(s.to_numpy(dtype=float))

    return pd.DataFrame({
        "date": s.index,
        "raw_rel": s.to_numpy(dtype=float),
    })


def segment_prediction(pred_df, penalty):
    y = pred_df["raw_rel"].to_numpy(dtype=float)
    spans = l2_segment(
        y,
        penalty=float(penalty),
        min_len=MIN_STATE_DAYS,
    )

    out = pred_df.copy()
    out["cluster"] = -1
    out["state_rel"] = np.nan

    rows = []

    for k, (i, j) in enumerate(spans):
        level = float(np.median(y[i:j]))
        out.loc[i:j-1, "cluster"] = k
        out.loc[i:j-1, "state_rel"] = level

        rows.append({
            "cluster": k,
            "start": out.loc[i, "date"],
            "end": out.loc[j-1, "date"],
            "days": j - i,
            "state_rel": level,
        })

    # Keep the temporal component day-weighted zero mean.
    mean_rel = float(np.average(out["state_rel"]))
    out["state_rel"] = out["state_rel"] - mean_rel

    states = pd.DataFrame(rows)
    states["state_rel"] = states["state_rel"] - mean_rel

    return out, states


def score_holdout(holdout, train_ids, alpha, penalty):
    model = fit_ridge(train_ids, alpha)
    raw_pred = raw_temporal_prediction(model, holdout)
    seg_pred, states = segment_prediction(raw_pred, penalty)

    L = physics_level(holdout, train_ids)

    truth = label_state_cache[holdout]
    scored = truth[truth["scored"]].merge(
        seg_pred[["date", "state_rel", "cluster"]],
        on="date",
        how="inner",
    )

    scored["pred"] = L + scored["state_rel"]
    scored["pred_const"] = L

    err = scored["pred"] - scored["target"]
    err0 = scored["pred_const"] - scored["target"]

    return {
        "mae": float(err.abs().mean()),
        "rmse": float(np.sqrt(np.mean(err.to_numpy() ** 2))),
        "constant_mae": float(err0.abs().mean()),
        "constant_rmse": float(np.sqrt(np.mean(err0.to_numpy() ** 2))),
        "n_states": int(len(states)),
    }



## 3. Strict nested leave-one-turbine-out validation

For each outer holdout turbine:

- its labels are completely hidden;
- the other two labelled turbines choose the regularization strength and state-segmentation penalty;
- the model is then fitted on those two turbines and transferred to the held-out turbine;
- the held-out turbine's absolute yaw centre comes from the original physics calibration.

This is the acceptance test. The new state-aware model must beat the constant physics baseline **across held-out turbines**, not merely fit the labelled days it trained on.


In [ ]:

def inner_score(train_ids, alpha, penalty):
    maes = []

    for inner_holdout in train_ids:
        inner_train = [
            t for t in train_ids
            if t != inner_holdout
        ]

        result = score_holdout(
            inner_holdout,
            inner_train,
            alpha,
            penalty,
        )
        maes.append(result["mae"])

    return float(np.mean(maes))


outer_rows = []

for holdout in TRAIN:
    train_ids = [
        t for t in TRAIN
        if t != holdout
    ]

    candidates = []

    for alpha in ALPHA_GRID:
        for penalty in SEG_PENALTY_GRID:
            candidates.append({
                "alpha": alpha,
                "penalty": penalty,
                "inner_mae": inner_score(
                    train_ids,
                    alpha,
                    penalty,
                ),
            })

    candidate_table = pd.DataFrame(candidates)

    best = candidate_table.sort_values(
        ["inner_mae", "alpha", "penalty"]
    ).iloc[0]

    result = score_holdout(
        holdout,
        train_ids,
        float(best["alpha"]),
        float(best["penalty"]),
    )

    outer_rows.append({
        "holdout": holdout,
        "alpha": float(best["alpha"]),
        "penalty": float(best["penalty"]),
        "n_states": result["n_states"],
        "constant_mae": result["constant_mae"],
        "state_mae": result["mae"],
        "constant_rmse": result["constant_rmse"],
        "state_rmse": result["rmse"],
    })


outer = pd.DataFrame(outer_rows)

display(outer.round(3))

CONST_MACRO_MAE = float(outer["constant_mae"].mean())
STATE_MACRO_MAE = float(outer["state_mae"].mean())

CONST_MACRO_RMSE = float(outer["constant_rmse"].mean())
STATE_MACRO_RMSE = float(outer["state_rmse"].mean())

print("Nested LOTO macro MAE")
print("  constant:", CONST_MACRO_MAE)
print("  state   :", STATE_MACRO_MAE)

print("\nNested LOTO macro RMSE")
print("  constant:", CONST_MACRO_RMSE)
print("  state   :", STATE_MACRO_RMSE)

MODEL_ACCEPTED = STATE_MACRO_MAE < CONST_MACRO_MAE

print("\nMODEL_ACCEPTED:", MODEL_ACCEPTED)



## 4. Final hyperparameters and unseen turbines

The final regularization and state penalty are selected using only the three labelled turbines in turbine-level cross-validation.

No PPP_WTG17 or SSS_WTG06 labels are used.


In [ ]:

grid_rows = []

for alpha in ALPHA_GRID:
    for penalty in SEG_PENALTY_GRID:
        fold_maes = []

        for holdout in TRAIN:
            train_ids = [
                t for t in TRAIN
                if t != holdout
            ]

            r = score_holdout(
                holdout,
                train_ids,
                alpha,
                penalty,
            )
            fold_maes.append(r["mae"])

        grid_rows.append({
            "alpha": alpha,
            "penalty": penalty,
            "macro_mae": float(np.mean(fold_maes)),
        })

grid = pd.DataFrame(grid_rows).sort_values(
    ["macro_mae", "alpha", "penalty"]
)

best = grid.iloc[0]

FINAL_ALPHA = float(best["alpha"])
FINAL_PENALTY = float(best["penalty"])

print("FINAL_ALPHA:", FINAL_ALPHA)
print("FINAL_PENALTY:", FINAL_PENALTY)

final_temporal_model = fit_ridge(
    TRAIN,
    FINAL_ALPHA,
)


def deploy_target(turbine_id):
    raw_pred = raw_temporal_prediction(
        final_temporal_model,
        turbine_id,
    )

    pred, states = segment_prediction(
        raw_pred,
        FINAL_PENALTY,
    )

    L = FROZEN_TARGET_LEVEL[turbine_id]

    pred["yaw_misalignment_deg"] = (
        L + pred["state_rel"]
    )

    pred["turbine_id"] = turbine_id

    return {
        "global_level": L,
        "states": states,
        "prediction": pred[
            [
                "turbine_id",
                "date",
                "yaw_misalignment_deg",
                "cluster",
                "state_rel",
            ]
        ],
    }


validate_model = deploy_target(VALIDATE_TARGET)
final_model = deploy_target(FINAL_TARGET)

print("\nPPP_WTG17")
print("global level:", validate_model["global_level"])
print("states:", len(validate_model["states"]))
display(validate_model["states"].round(3))

print("\nSSS_WTG06")
print("global level:", final_model["global_level"])
print("states:", len(final_model["states"]))
display(final_model["states"].round(3))



## 5. Candidate files

A candidate file is written **only if strict nested turbine-level validation beats the constant baseline**.

The final candidate keeps a research filename until the organizer's replacement-final naming convention is confirmed.


In [ ]:

OUT_DIR = ROOT / "candidate_outputs"
OUT_DIR.mkdir(exist_ok=True)


def submission_frame(deployment):
    out = deployment["prediction"].copy()

    out["date"] = pd.to_datetime(
        out["date"]
    ).dt.strftime("%Y-%m-%d")

    out["cluster"] = out["cluster"].astype(int)

    out = out[
        [
            "turbine_id",
            "date",
            "yaw_misalignment_deg",
            "cluster",
        ]
    ]

    assert len(out) == 731, f"Expected 731 rows, got {len(out)}"
    assert out["yaw_misalignment_deg"].notna().all()
    assert np.isfinite(out["yaw_misalignment_deg"]).all()
    assert out["yaw_misalignment_deg"].abs().max() <= 90

    return out


if MODEL_ACCEPTED:
    val_out = submission_frame(validate_model)
    fin_out = submission_frame(final_model)

    val_path = OUT_DIR / "Results_33_T3_2.csv"
    fin_path = (
        OUT_DIR
        / "SSS_WTG06_daily_feature_state_candidate.csv"
    )

    val_out.to_csv(
        val_path,
        index=False,
        float_format="%.6f",
    )

    fin_out.to_csv(
        fin_path,
        index=False,
        float_format="%.6f",
    )

    print("WROTE:", val_path)
    print("WROTE:", fin_path)
else:
    print("NOT WRITING SUBMISSION FILES.")
    print(
        "Reason: strict turbine-level validation "
        "did not beat the constant baseline."
    )



## Send back only these outputs

- the `outer` table;
- `MODEL_ACCEPTED`;
- `FINAL_ALPHA` and `FINAL_PENALTY`;
- the PPP_WTG17 state table;
- the SSS_WTG06 state table.

This notebook is deliberately much shorter than the consensus experiments. If it fails strict LOTO, the result is enough to reject the feature-based temporal model without another round of manual parameter tuning.
